# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.crm_sales")

# Read from bronze

In [0]:
try:
    df = spark.table("workspace.bronze.crm_sales_details_raw")
except Exception as e:
    logger.error(f"Failed to read Bronze table: {e}")
    raise

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## Casting data types

In [0]:
df = (
    df.withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("sales_amount", F.col("sales_amount").cast(IntegerType()))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("price", F.col("price").cast(IntegerType()))
)

In [0]:
date_fields = ["order_date", "ship_date", "due_date"]

for field in date_fields:
    df = df.withColumn(
        field,
        F.when(
            F.length(F.col(field)) != 8,
            None
        ).otherwise(F.to_date(F.col(field), "yyyyMMdd"))
    )

## Price corrections

In [0]:
price_invalid = (F.col("price").isNull()) | (F.col("price") <= 0)

df = (
    df.withColumn(
        "price",
        F.when(
            price_invalid & (F.col("quantity") != 0),
            F.col("sales_amount") / F.col("quantity")
        ).otherwise(F.col("price"))
    )
)

# Write into Silver Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")
except Exception as e:
    logger.error(f"Failed to write Silver table: {e}")
    raise